In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, r2_score
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('All libraries imported successfully!')

In [ ]:
DATA_DIR = './data'         
YEARS    = range(2008, 2019) 
datasets = {}
for year in YEARS:
    path = os.path.join(DATA_DIR, f'vehicles_{year}.csv')
    if os.path.exists(path):
        df = pd.read_csv(path, low_memory=False)
        datasets[year] = df
        print(f'{year}: {df.shape[0]:,} rows, {df.shape[1]} columns')
    else:
        print(f'[WARNING] File not found: {path}')

print(f'\nLoaded {len(datasets)} datasets.')

---
# Problem 1 — Exploratory Data Analysis (EDA)

In [ ]:
sample_counts = {year: df.shape[0] for year, df in datasets.items()}
sample_df = pd.DataFrame.from_dict(sample_counts, orient='index', columns=['Number of Samples'])
sample_df.index.name = 'Year'
print(sample_df.to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(sample_df.index, sample_df['Number of Samples'], color='steelblue', edgecolor='white')
ax.set_title('Number of Vehicle Samples per Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Samples')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
col_counts = {year: df.shape[1] for year, df in datasets.items()}
print(pd.DataFrame.from_dict(col_counts, orient='index', columns=['Number of Columns']).to_string())

In [ ]:
dup_counts = {year: df.duplicated().sum() for year, df in datasets.items()}
print(pd.DataFrame.from_dict(dup_counts, orient='index', columns=['Duplicate Rows']).to_string())

In [ ]:
first_year = min(datasets.keys())
print(f'--- Datatypes for {first_year} ---')
print(datasets[first_year].dtypes.to_string())

In [ ]:
for year, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    pct     = (missing / len(df) * 100).round(2)
    result  = pd.DataFrame({'Missing Count': missing, 'Missing %': pct})
    print(f'\n=== {year} — {len(result)} columns with missing values ===')
    if len(result) > 0:
        print(result.to_string())
    else:
        print('  No missing values.')

In [ ]:
for year, df in datasets.items():
    unique_counts = df.nunique(dropna=True)
    print(f'\n=== {year} ===')
    print(unique_counts.to_string())

In [ ]:
df_sample = datasets[first_year]
cat_cols = df_sample.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    print(f'\n--- {col} ---')
    print(df_sample[col].value_counts(dropna=True).head(10).to_string())

In [ ]:
def find_col(df, keywords):
    """Return the first column whose name contains any keyword (case-insensitive)."""
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in col.lower():
                return col
    return None

df_sample = datasets[first_year]

veh_class_col = find_col(df_sample, ['VClass', 'vehicle class', 'class'])
make_col      = find_col(df_sample, ['make', 'manufacturer'])
city_mpg_col  = find_col(df_sample, ['city08', 'city mpg', 'cityMpg'])
hwy_mpg_col   = find_col(df_sample, ['highway08', 'hwy mpg', 'highwayMpg'])
comb_mpg_col  = find_col(df_sample, ['comb08', 'combined'])

print('Detected columns:')
print(f'  Vehicle Class : {veh_class_col}')
print(f'  Make          : {make_col}')
print(f'  City MPG      : {city_mpg_col}')
print(f'  Highway MPG   : {hwy_mpg_col}')
print(f'  Combined MPG  : {comb_mpg_col}')

if veh_class_col and city_mpg_col:
    grouped = (df_sample.groupby(veh_class_col)[city_mpg_col]
               .agg(['mean', 'median', 'count'])
               .round(2)
               .sort_values('mean', ascending=False))
    print(f'\n--- Average City MPG by Vehicle Class ({first_year}) ---')
    print(grouped.to_string())

In [ ]:
alt_fuel_counts = {}

for year, df in datasets.items():
    fuel_col = find_col(df, ['fuelType1', 'fuel type', 'fuelType'])
    if fuel_col:
        alt = df[~df[fuel_col].str.lower().str.contains('gasoline|diesel', na=False)]
        alt_fuel_counts[year] = alt.shape[0]
    else:
        alt_fuel_counts[year] = np.nan

alt_df = pd.Series(alt_fuel_counts, name='Alt Fuel Models').dropna()

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#e74c3c' if y == 2018 else '#3498db' for y in alt_df.index]
ax.bar(alt_df.index, alt_df.values, color=colors, edgecolor='white', linewidth=0.8)
ax.set_title('Number of Models Using Alternative Fuels (2008–2018)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Models')
ax.set_xticks(list(alt_df.index))
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            int(bar.get_height()), ha='center', va='bottom', fontsize=9)
ax.legend(handles=[
    plt.Rectangle((0,0),1,1, color='#e74c3c', label='2018'),
    plt.Rectangle((0,0),1,1, color='#3498db', label='Other Years')
], loc='upper left')
plt.tight_layout()
plt.show()

if 2018 in alt_df.index and 2008 in alt_df.index:
    diff = alt_df[2018] - alt_df[2008]
    print(f'\nAlt-fuel models in 2008: {int(alt_df[2008])}')
    print(f'Alt-fuel models in 2018: {int(alt_df[2018])}')
    print(f'Increase: {int(diff)} more models ({diff/alt_df[2008]*100:.1f}%)')

In [ ]:
class_mpg_by_year = []

for year, df in datasets.items():
    vc_col  = find_col(df, ['VClass', 'vehicle class', 'class'])
    mpg_col = find_col(df, ['comb08', 'combined', 'combMpg'])
    if vc_col and mpg_col:
        g = df.groupby(vc_col)[mpg_col].mean().reset_index()
        g.columns = ['VehicleClass', 'CombMPG']
        g['Year'] = year
        class_mpg_by_year.append(g)

if class_mpg_by_year:
    class_mpg_df = pd.concat(class_mpg_by_year, ignore_index=True)

    # Keep top vehicle classes by count
    top_classes = (class_mpg_df.groupby('VehicleClass')['CombMPG']
                   .count().nlargest(8).index.tolist())
    class_mpg_df = class_mpg_df[class_mpg_df['VehicleClass'].isin(top_classes)]

    fig, ax = plt.subplots(figsize=(14, 7))
    palette = sns.color_palette('tab10', n_colors=len(top_classes))
    for i, vc in enumerate(top_classes):
        sub = class_mpg_df[class_mpg_df['VehicleClass'] == vc].sort_values('Year')
        ax.plot(sub['Year'], sub['CombMPG'], marker='o', label=vc, color=palette[i], linewidth=2)

    ax.set_title('Average Combined MPG by Vehicle Class (2008–2018)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Average Combined MPG')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
smartway_rows = []

for year, df in datasets.items():
    sw_col  = find_col(df, ['smartway', 'SmartWay', 'smart way'])
    mpg_col = find_col(df, ['comb08', 'combined'])
    ghg_col = find_col(df, ['co2', 'ghg', 'greenhouse'])
    if sw_col:
        sw_df = df[df[sw_col].astype(str).str.strip().isin(['Yes', 'Elite', 'Y', '1', 'yes'])].copy()
        row = {'Year': year, 'SmartWay Count': len(sw_df)}
        if mpg_col:
            row['Avg Comb MPG'] = sw_df[mpg_col].mean()
        if ghg_col:
            row['Avg GHG'] = pd.to_numeric(sw_df[ghg_col], errors='coerce').mean()
        smartway_rows.append(row)

if smartway_rows:
    sw_summary = pd.DataFrame(smartway_rows).set_index('Year')
    print('SmartWay Vehicle Summary:')
    print(sw_summary.round(2).to_string())

    fig, axes = plt.subplots(1, min(3, len(sw_summary.columns)), figsize=(15, 5))
    if not isinstance(axes, np.ndarray): axes = [axes]

    for ax, col in zip(axes, sw_summary.columns):
        if col in sw_summary.columns:
            ax.plot(sw_summary.index, sw_summary[col], marker='o', color='#27ae60', linewidth=2)
            ax.fill_between(sw_summary.index, sw_summary[col], alpha=0.1, color='#27ae60')
            ax.set_title(col, fontsize=12, fontweight='bold')
            ax.set_xlabel('Year')
    plt.suptitle('SmartWay Vehicle Trends (2008–2018)', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('SmartWay column not detected — check column names in your datasets.')

In [ ]:
df_s = datasets[first_year]
num_cols = df_s.select_dtypes(include=[np.number]).columns.tolist()
mpg_col  = find_col(df_s, ['comb08', 'combined'])

if mpg_col and mpg_col in num_cols:
    corr = df_s[num_cols].corr()[mpg_col].drop(mpg_col).sort_values()

    fig, ax = plt.subplots(figsize=(10, max(6, len(corr) * 0.3)))
    colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr.values]
    ax.barh(corr.index, corr.values, color=colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Feature Correlation with Combined MPG ({first_year})', fontsize=14, fontweight='bold')
    ax.set_xlabel('Pearson Correlation')
    plt.tight_layout()
    plt.show()

    print('Top positive correlates with MPG:')
    print(corr.tail(5).round(3).to_string())
    print('\nTop negative correlates with MPG:')
    print(corr.head(5).round(3).to_string())

In [ ]:
mpg_data = []
for year, df in datasets.items():
    mpg_col = find_col(df, ['comb08', 'combined'])
    if mpg_col:
        vals = pd.to_numeric(df[mpg_col], errors='coerce').dropna()
        vals = vals[(vals > 0) & (vals < 200)]  # remove outliers/electric-only
        mpg_data.append({'Year': year, 'Comb MPG': vals})

if mpg_data:
    fig, ax = plt.subplots(figsize=(14, 6))
    positions = sorted([d['Year'] for d in mpg_data])
    data_to_plot = [d['Comb MPG'].values for d in sorted(mpg_data, key=lambda x: x['Year'])]
    bp = ax.boxplot(data_to_plot, positions=positions, widths=0.6,
                    patch_artist=True, showfliers=False,
                    medianprops=dict(color='red', linewidth=2))
    for patch in bp['boxes']:
        patch.set_facecolor('#85c1e9')
    ax.set_title('Distribution of Combined MPG per Year (2008–2018)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Combined MPG')
    ax.set_xticks(positions)
    plt.tight_layout()
    plt.show()

---
# Problem 3 — Machine Learning Models

In [ ]:
# ── Find common columns across all datasets ───────────────────────────────────
all_col_sets = [set(df.columns) for df in datasets.values()]
common_cols = set.intersection(*all_col_sets)
print(f'Common columns across all years: {len(common_cols)}')
print(sorted(common_cols))

In [ ]:
# ── Build merged dataframe ────────────────────────────────────────────────────
merged_dfs = []
for year, df in datasets.items():
    subset = df[list(common_cols)].copy()
    subset['Year'] = year
    merged_dfs.append(subset)

merged = pd.concat(merged_dfs, ignore_index=True)
print(f'Merged dataset shape: {merged.shape}')
print(f'Years present: {sorted(merged["Year"].unique())}')

In [ ]:
# ── Ensure numeric columns have consistent dtypes ─────────────────────────────
num_suspects = merged.select_dtypes(include='object').columns.tolist()

for col in num_suspects:
    try:
        converted = pd.to_numeric(merged[col], errors='coerce')
        if converted.notna().mean() > 0.5:   # more than 50% parseable → numeric
            merged[col] = converted
    except Exception:
        pass

print('Dtypes after conversion:')
print(merged.dtypes.value_counts())

In [ ]:
# ── Drop duplicate rows from merged ──────────────────────────────────────────
before = len(merged)
merged.drop_duplicates(inplace=True)
print(f'Removed {before - len(merged)} duplicates. Final shape: {merged.shape}')
merged.head(3)

In [ ]:
# Select numeric features for clustering
cluster_features = merged.select_dtypes(include=[np.number]).columns.tolist()
cluster_features = [c for c in cluster_features if c != 'Year']

cluster_df = merged[cluster_features].dropna()
print(f'Using {len(cluster_features)} numeric features, {len(cluster_df):,} rows for clustering')
print(cluster_features)

In [ ]:
# ── Scale features ────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_cluster = scaler.fit_transform(cluster_df)

# Subsample if too large (for speed)
MAX_CLUSTER_ROWS = 10_000
if len(X_cluster) > MAX_CLUSTER_ROWS:
    np.random.seed(42)
    idx = np.random.choice(len(X_cluster), MAX_CLUSTER_ROWS, replace=False)
    X_cluster_sample = X_cluster[idx]
    print(f'Subsampled to {MAX_CLUSTER_ROWS:,} rows for clustering speed.')
else:
    X_cluster_sample = X_cluster

# ── Elbow Method ──────────────────────────────────────────────────────────────
inertia_values = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster_sample)
    inertia_values.append(km.inertia_)
    print(f'  k={k:2d}  inertia={km.inertia_:,.0f}')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(list(K_range), inertia_values, marker='o', linewidth=2, color='#e74c3c')
ax.set_title('Elbow Method — KMeans Inertia vs Number of Clusters', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia (within-cluster SSE)')
ax.set_xticks(list(K_range))
plt.tight_layout()
plt.show()

In [ ]:
# ── Fit optimal KMeans (adjust OPTIMAL_K based on elbow above) ───────────────
OPTIMAL_K = 4   # <-- Change this after inspecting the elbow plot

km_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
cluster_df['Cluster'] = km_final.fit_predict(X_cluster)

print(f'\nCluster sizes:')
print(cluster_df['Cluster'].value_counts().sort_index().to_string())

In [ ]:
# ── Cluster Profiles ──────────────────────────────────────────────────────────
print('Cluster Mean Profiles:')
print(cluster_df.groupby('Cluster').mean().round(2).T.to_string())

In [ ]:
# ── Visualise clusters using first 2 principal features ──────────────────────
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster)

fig, ax = plt.subplots(figsize=(9, 6))
palette = sns.color_palette('Set2', OPTIMAL_K)
for c in range(OPTIMAL_K):
    mask = cluster_df['Cluster'] == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], s=5, alpha=0.4,
               color=palette[c], label=f'Cluster {c}')
ax.set_title('KMeans Clusters (PCA-reduced to 2D)', fontsize=13, fontweight='bold')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend(markerscale=4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Identify SmartWay column ──────────────────────────────────────────────────
sw_col = find_col(merged, ['smartway', 'SmartWay'])

if sw_col is None:
    print('SmartWay column not found — check column names!')
else:
    print(f'SmartWay column: "{sw_col}"')
    print(merged[sw_col].value_counts())

In [ ]:
# ── Encode SmartWay as binary target (0 = No, 1 = Yes / Elite) ───────────────
if sw_col:
    sw_map = {v: (0 if str(v).strip().lower() in ['no', '0', 'nan', ''] else 1)
              for v in merged[sw_col].unique()}
    print('SmartWay mapping:', sw_map)

    merged['SmartWay_Binary'] = merged[sw_col].map(sw_map).fillna(0).astype(int)
    print('\nTarget class distribution:')
    print(merged['SmartWay_Binary'].value_counts())

In [ ]:
# ── Build feature matrix for classification ───────────────────────────────────
if sw_col:
    exclude_cols = [sw_col, 'SmartWay_Binary']
    feat_cols = merged.select_dtypes(include=[np.number]).columns.tolist()
    feat_cols = [c for c in feat_cols if c not in exclude_cols]

    clf_data = merged[feat_cols + ['SmartWay_Binary']].dropna()
    X = clf_data[feat_cols].values
    y = clf_data['SmartWay_Binary'].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler_clf = StandardScaler()
    X_train_sc = scaler_clf.fit_transform(X_train)
    X_test_sc  = scaler_clf.transform(X_test)

    print(f'Training samples : {X_train.shape[0]:,}')
    print(f'Test samples     : {X_test.shape[0]:,}')
    print(f'Feature count    : {X_train.shape[1]}')

In [ ]:
# ── Logistic Regression ───────────────────────────────────────────────────────
if sw_col:
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_train_sc, y_train)
    y_pred_lr = lr.predict(X_test_sc)

    print('=== Logistic Regression ===')
    print(f'Accuracy : {accuracy_score(y_test, y_pred_lr):.4f}')
    print(classification_report(y_test, y_pred_lr, target_names=['Not SmartWay', 'SmartWay']))

In [ ]:
# ── Decision Tree ─────────────────────────────────────────────────────────────
if sw_col:
    dt = DecisionTreeClassifier(max_depth=6, random_state=42)
    dt.fit(X_train, y_train)   # Decision tree doesn't need scaling
    y_pred_dt = dt.predict(X_test)

    print('=== Decision Tree ===')
    print(f'Accuracy : {accuracy_score(y_test, y_pred_dt):.4f}')
    print(classification_report(y_test, y_pred_dt, target_names=['Not SmartWay', 'SmartWay']))

In [ ]:
# ── Confusion Matrices Side-by-Side ──────────────────────────────────────────
if sw_col:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    labels = ['Not SmartWay', 'SmartWay']

    for ax, y_pred, title in zip(
        axes,
        [y_pred_lr, y_pred_dt],
        ['Logistic Regression', 'Decision Tree']
    ):
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=labels, yticklabels=labels, ax=ax)
        ax.set_title(f'Confusion Matrix — {title}', fontweight='bold')
        ax.set_ylabel('True Label')
        ax.set_xlabel('Predicted Label')

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Decision Tree Feature Importances ────────────────────────────────────────
if sw_col:
    fi = pd.Series(dt.feature_importances_, index=feat_cols).sort_values(ascending=False)
    print('Top 10 features for Decision Tree:')
    print(fi.head(10).round(4).to_string())

    fig, ax = plt.subplots(figsize=(10, 5))
    fi.head(15).sort_values().plot.barh(ax=ax, color='#2980b9')
    ax.set_title('Decision Tree — Top 15 Feature Importances', fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Model Comparison ──────────────────────────────────────────────────────────
if sw_col:
    lr_acc = accuracy_score(y_test, y_pred_lr)
    dt_acc = accuracy_score(y_test, y_pred_dt)

    print('\n===== MODEL COMPARISON (SmartWay Classification) =====')
    print(f'  Logistic Regression Accuracy : {lr_acc:.4f}')
    print(f'  Decision Tree Accuracy       : {dt_acc:.4f}')
    best = 'Logistic Regression' if lr_acc > dt_acc else 'Decision Tree'
    print(f'\nBest Model: {best}')
    print("""
Reasoning:
- Logistic Regression is better when features are roughly linearly separable and gives 
  probabilistic outputs which can be useful.
- Decision Tree captures non-linear boundaries, is interpretable, and doesn't need 
  feature scaling. However, it can overfit without proper depth constraints.
- Compare precision/recall for the minority class (SmartWay=1) since the classes 
  may be imbalanced — a higher F1-score for that class is more meaningful than 
  overall accuracy alone.
""")

In [ ]:
city_col = find_col(merged, ['city08', 'cityMpg', 'city mpg'])

if city_col is None:
    print('City MPG column not found — check column names!')
else:
    print(f'City MPG column: "{city_col}"')

    # Drop the target and highly correlated MPG columns from features
    drop_for_reg = [city_col]
    for kw in ['hwy', 'highway', 'comb', 'combined', 'unadjusted', 'city']:
        col = find_col(merged, [kw])
        if col and col != city_col:
            drop_for_reg.append(col)

    reg_num_cols = merged.select_dtypes(include=[np.number]).columns.tolist()
    reg_feat_cols = [c for c in reg_num_cols if c not in drop_for_reg and c != 'Year']

    reg_data = merged[reg_feat_cols + [city_col]].dropna()
    reg_data = reg_data[reg_data[city_col] > 0]  # remove zero/invalid

    X_reg = reg_data[reg_feat_cols].values
    y_reg = reg_data[city_col].values

    X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
    print(f'Train: {X_tr.shape[0]:,} rows, Test: {X_te.shape[0]:,} rows')

In [ ]:
# ── Linear Regression ─────────────────────────────────────────────────────────
if city_col:
    sc_reg = StandardScaler()
    X_tr_sc = sc_reg.fit_transform(X_tr)
    X_te_sc = sc_reg.transform(X_te)

    lin_reg = LinearRegression()
    lin_reg.fit(X_tr_sc, y_tr)
    y_pred_lin = lin_reg.predict(X_te_sc)

    print('=== Linear Regression ===')
    print(f'R² Score : {r2_score(y_te, y_pred_lin):.4f}')
    print(f'RMSE     : {np.sqrt(mean_squared_error(y_te, y_pred_lin)):.4f}')

In [ ]:
# ── Random Forest Regression (captures non-linear relationships) ──────────────
if city_col:
    rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_reg.fit(X_tr, y_tr)
    y_pred_rf = rf_reg.predict(X_te)

    print('=== Random Forest Regressor ===')
    print(f'R² Score : {r2_score(y_te, y_pred_rf):.4f}')
    print(f'RMSE     : {np.sqrt(mean_squared_error(y_te, y_pred_rf)):.4f}')

In [ ]:
# ── Actual vs Predicted Plot ──────────────────────────────────────────────────
if city_col:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, y_pred, title in zip(
        axes,
        [y_pred_lin, y_pred_rf],
        ['Linear Regression', 'Random Forest']
    ):
        ax.scatter(y_te, y_pred, alpha=0.3, s=10, color='#2980b9')
        lims = [min(y_te.min(), y_pred.min()), max(y_te.max(), y_pred.max())]
        ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Fit')
        ax.set_title(f'{title}\nR²={r2_score(y_te, y_pred):.3f}', fontweight='bold')
        ax.set_xlabel('Actual City MPG')
        ax.set_ylabel('Predicted City MPG')
        ax.legend()

    plt.suptitle('Actual vs Predicted City MPG', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Random Forest Feature Importances ────────────────────────────────────────
if city_col:
    rf_fi = pd.Series(rf_reg.feature_importances_, index=reg_feat_cols).sort_values(ascending=False)
    print('Top 10 features for predicting City MPG:')
    print(rf_fi.head(10).round(4).to_string())

    fig, ax = plt.subplots(figsize=(10, 5))
    rf_fi.head(15).sort_values().plot.barh(ax=ax, color='#8e44ad')
    ax.set_title('Random Forest — Top 15 Features for City MPG Prediction', fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()